# Virtual Manipulation of GRIB2 with VirtualiZarr and grib2io

This notebook demonstrates how to use `VirtualiZarr` to open a grib2io kerchunk manifest
as a virtual dataset — holding only metadata references, no data bytes — and then
load actual data through the standard `zarr` engine (backed by `fsspec` and our custom global `numcodecs` decoders) or open the original GRIB2 file directly using the `grib2io` backend.


## Notebook Convention

Use the standard grib2io xarray backend interface or the standard zarr backend with fsspec reference filesystem.

In [1]:
import xarray as xr
import grib2io
import json
import pathlib
import os
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
from grib2io.kerchunk import ReferenceGenerator

# Optional notebook-only dependency: do not require this in grib2io package deps.
try:
    from obstore.store import LocalStore
    from obspec_utils.registry import ObjectStoreRegistry

    OBSTORE_AVAILABLE = True
except ImportError:
    OBSTORE_AVAILABLE = False
    print("Optional packages not installed: obstore, obspec-utils. Install them in this notebook to enable registry-backed VirtualiZarr access.")

/opt/homebrew/Caskroom/miniforge/base/envs/grib2io/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Build a Kerchunk Manifest

`ReferenceGenerator` scans a GRIB2 file and produces a kerchunk v1 reference manifest —
a plain dict mapping variable/coordinate names to byte-range references. No data is read.


In [2]:
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Manifest: {len(manifest['refs'])} refs for {grib2_file}")

# Persist for VirtualiZarr to read via URI
manifest_path = pathlib.Path(grib2_file).with_suffix(".json")
manifest_path.write_text(json.dumps(manifest))

Manifest: 1231 refs for /Users/barry/Documents/grib2io/tests/input_data/gfs.t00z.pgrb2.1p00.f024


1685140

## 2. Open as a VirtualiZarr Dataset

`open_virtual_dataset` reads the manifest and returns an `xarray.Dataset` backed by
`ManifestArray` objects — virtual metadata only, no data downloaded. This is useful
for inspecting structure, combining datasets, or generating consolidated manifests.


In [3]:
manifest_url = manifest_path.resolve().as_uri()

from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry

registry = ObjectStoreRegistry(
    {
        manifest_url: LocalStore(prefix=str(manifest_path.parent)),
    }
)

vds = open_virtual_dataset(
    url=manifest_url,
    registry=registry,
    parser=KerchunkJSONParser(),
    loadable_variables=[],
)

display(vds)

<xarray.Dataset> Size: 195MB
Dimensions:                              (valid_time: 1, mean_sea_level: 1,
                                          y: 181, x: 360, hybrid_level: 1,
                                          hybrid_level_2: 2,
                                          entire_atmosphere: 1, surface: 1,
                                          planetary_boundary_layer: 1,
                                          isobaric_surface: 41,
                                          ...
                                          highest_tropospheric_freezing_level: 1,
                                          pressure_difference_from_ground: 1,
                                          pressure_difference_from_ground_2: 3,
                                          sigma_level: 4, sigma_level_2: 1,
                                          pressure_difference_from_ground_3: 1,
                                          potential_vorticity_surface: 2)
Coordinates: (12/49)
    valid_time                           (valid_time) int64 8B ManifestArray<...
    mean_sea_level                       (mean_sea_level) float64 8B Manifest...
    latitude                             (y, x) float64 521kB ManifestArray<s...
    longitude                            (y, x) float64 521kB ManifestArray<s...
    hybrid_level                         (hybrid_level) float64 8B ManifestAr...
    hybrid_level_2                       (hybrid_level_2) float64 16B Manifes...
    ...                                   ...
    pressure_difference_from_ground      (pressure_difference_from_ground) float64 8B ManifestArray<shape=(1,), dtype=float64, chu...
    pressure_difference_from_ground_2    (pressure_difference_from_ground_2) float64 24B ManifestArray<shape=(3,), dtype=float64, ...
    sigma_level                          (sigma_level) float64 32B ManifestAr...
    sigma_level_2                        (sigma_level_2) float64 8B ManifestA...
    pressure_difference_from_ground_3    (pressure_difference_from_ground_3) float64 8B ManifestArray<shape=(1,), dtype=float64, c...
    potential_vorticity_surface          (potential_vorticity_surface) float64 16B ManifestArray<shape=(2,), dtype=float64, chunks...
Dimensions without coordinates: y, x
Data variables: (12/170)
    PRMSL                                (valid_time, mean_sea_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, ...
    CLMR                                 (valid_time, hybrid_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, 1,...
    ICMR                                 (valid_time, hybrid_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, 1,...
    RWMR                                 (valid_time, hybrid_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, 1,...
    SNMR                                 (valid_time, hybrid_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, 1,...
    GRLE                                 (valid_time, hybrid_level, y, x) float32 261kB ManifestArray<shape=(1, 1, 181, 360), dtype=float32, chunks=(1, 1,...
    ...                                   ...
    UGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ManifestArray<shape=(1, 2, 181, 360), dtype=float32...
    VGRD_8                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ManifestArray<shape=(1, 2, 181, 360), dtype=float32...
    TMP_11                               (valid_time, potential_vorticity_surface, y, x) float32 521kB ManifestArray<shape=(1, 2, 181, 360), dtype=float32...
    HGT_7                                (valid_time, potential_vorticity_surface, y, x) float32 521kB ManifestArray<shape=(1, 2, 181, 360), dtype=float32...
    PRES_12                              (valid_time, potential_vorticity_surface, y, x) floa

## 3. Load Actual Data

A `ManifestArray` is virtual, so calling `.compute()` on a VirtualiZarr dataset raises
an error. To load real data, use the standard xarray interfaces:

- Open a manifest (Kerchunk reference file) with `xr.open_dataset(mapper, engine="zarr")` using standard fsspec
- Open a GRIB2 file directly with `xr.open_dataset(grib2_file, engine="grib2io")`

In [4]:
# Option A: Open the manifest (Kerchunk reference file) directly using xarray with engine="zarr" and fsspec
import fsspec

fs = fsspec.filesystem("reference", fo=str(manifest_path))
mapper = fs.get_mapper("")
ds = xr.open_dataset(mapper, engine="zarr", consolidated=False)

# Option B: Open the original GRIB2 file directly using grib2io standard backend
# ds = xr.open_dataset(grib2_file, engine="grib2io")

if "TMP" in ds.data_vars:
    data = ds.TMP.isel(y=slice(0, 10), x=slice(0, 10)).compute()
    print(f"TMP subset: shape={data.shape}, min={float(data.min()):.2f}, max={float(data.max()):.2f}")
    display(data)

TMP subset: shape=(1, 41, 10, 10), min=181.97, max=275.94


<xarray.DataArray 'TMP' (valid_time: 1, isobaric_surface: 41, y: 10, x: 10)> Size: 16kB
dask.array<getitem, shape=(1, 41, 10, 10), dtype=float32, chunksize=(1, 1, 10, 10), chunktype=numpy.ndarray>
Coordinates:
  * valid_time        (valid_time) datetime64[ns] 8B 2022-11-08
  * isobaric_surface  (isobaric_surface) float64 328B 1.0 2.0 ... 9.75e+04 1e+05
    latitude          (y, x) float64 800B dask.array<chunksize=(10, 10), meta=np.ndarray>
    longitude         (y, x) float64 800B dask.array<chunksize=(10, 10), meta=np.ndarray>
Dimensions without coordinates: y, x
Attributes:
    discipline:                0
    parameterCategory:         0
    parameterNumber:           0
    typeOfFirstFixedSurface:   100
    valueOfFirstFixedSurface:  1.0
    valid_time:                2022-11-08T00:00:00
    shortName:                 TMP
    fullName:                  Temperature
    units:                     K